如果说 `torch.einsum` 是运算（Math）的神器，那么 einops 就是张量变形（Structure）的神器。

PyTorch原生的 `view`、`reshape`、`transpose`、`permute`、`unsqueeze`、`squeeze` 常常让人头晕，因为它们依赖隐式的维度顺序（比如“第2维是啥来着？”）。

`einops` 的核心哲学是：不要让我猜，把维度的名字写出来！

它主要由三个“三巨头”函数组成：

1. **rearrange**：万能变形王（最常用）。
2. **reduce**：降维聚合（求平均、最大值等）。
3. **repeat**：维度复制。

### rearrange操作

1. 基础维度交换 (Permute)

In [8]:
from einops import rearrange, repeat
import torch

In [9]:
a = torch.randn(1, 3, 224, 224)
b = rearrange(a, "b c h w -> b h w c")
assert b.shape == (1, 224, 224, 3)

2. 维度融合 (Flatten / View)

In [10]:
a = torch.randn(1, 3, 196, 768)
b = rearrange(a, "b c h w -> b (h w c)")
assert b.shape == (1, 196 * 768 * 3,)

3. 维度拆分 (Unflatten / View)

In [11]:
a = torch.randn(2, 16, 768)
b = rearrange(a, "b (n h) d -> b n h d", h = 4)
assert b.shape == (2, 4, 4, 768)

图像分块 (Patchify - ViT 核心)
Vision Transformer 需要把图片切成小块（Patches）。
假设图片 [b, c, h, w]，patch 大小是 p x p。

In [12]:
a = torch.randn(1, 3, 224, 224)
b = rearrange(a, "b c (p1 h) (p2 w) -> b (h w) (c p1 p2)", p1=14, p2=14)
print(b.shape)

torch.Size([1, 256, 588])


### repeat操作
它是 expand、repeat、tile 的替代品。

广播 (Broadcasting)
假设你有一个类别向量 [batch, class_num]，你想把它扩展到每个像素上，以便和图像特征做拼接。
目标：[batch, class_num, h, w]。

In [15]:
a = torch.randn(1, 1, 768)
b = a.expand(32, -1, -1)
b1 = repeat(a, "1 1 d -> i 1 d", i=32)
assert torch.all(b==b1)